# Streaming ML in Production

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/streaming-ml/05-streaming-ml-in-production

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

**The problem.** The sketches, online learners, and detectors only pay off inside a reliable pipeline — and the pipeline's top failure mode is **training/serving skew** from features computed differently, or with a temporal leak, between training and serving. **The core idea:** features must be **point-in-time correct** — a training row for time `t` uses only values available before `t`. We implement an as-of join from scratch, then use the library version, and handle **delayed labels**.

In [ ]:
import numpy as np
import pandas as pd
np.random.seed(42)

## 1. From scratch — a point-in-time (as-of) feature join

We predict on transactions; a slow feature `acct_30d_avg` is updated over time in a history table. The value attached to a transaction at `event_time` must be the most recent update with `update_time <= event_time` — never a later one.

In [ ]:
txns = pd.DataFrame({
    'txn_id': [0, 1, 2, 3, 4, 5],
    'event_time': [10, 25, 40, 55, 70, 85],
})
feature_hist = pd.DataFrame({
    'update_time': [0, 30, 60, 90],
    'acct_30d_avg': [100.0, 180.0, 140.0, 220.0],
})

def asof_join_scratch(txns, hist):
    hist = hist.sort_values('update_time')
    vals = []
    for et in txns['event_time']:
        eligible = hist[hist['update_time'] <= et]      # only the past
        vals.append(eligible['acct_30d_avg'].iloc[-1] if len(eligible) else np.nan)
    out = txns.copy(); out['acct_30d_avg'] = vals
    return out

scratch = asof_join_scratch(txns, feature_hist)
print(scratch)

## 2. The library way — validate against pandas.merge_asof

`pandas.merge_asof` is the point-in-time join a feature store performs for you. We assert it matches our from-scratch result exactly.

In [ ]:
lib = pd.merge_asof(
    txns.sort_values('event_time'),
    feature_hist.sort_values('update_time'),
    left_on='event_time', right_on='update_time', direction='backward',
).sort_values('txn_id').reset_index(drop=True)

assert np.allclose(scratch['acct_30d_avg'], lib['acct_30d_avg']), \
    'from-scratch as-of join must match pandas.merge_asof'
print('from-scratch as-of join == pandas.merge_asof ✓')
print(lib[['txn_id', 'event_time', 'update_time', 'acct_30d_avg']])

## 3. Visualize it — the temporal leak

The dangerous shortcut is to join the *latest* feature value onto every row. That leaks the future: each training row sees information that did not exist at its own event time.

In [ ]:
naive_val = feature_hist['acct_30d_avg'].iloc[-1]        # 'just use the current value'
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.step(feature_hist['update_time'], feature_hist['acct_30d_avg'], where='post',
        color='#94a3b8', label='true feature history')
ax.scatter(scratch['event_time'], scratch['acct_30d_avg'], color='#14b8a6', zorder=3,
           label='point-in-time (correct)')
ax.scatter(txns['event_time'], [naive_val] * len(txns), color='#f43f5e', marker='x', zorder=3,
           label='naive latest-value (leaks future)')
ax.set_xlabel('time'); ax.set_ylabel('acct_30d_avg')
ax.set_title('Point-in-time vs future-leaking feature join', color='white')
ax.legend(fontsize=8); ax.grid(alpha=0.2); plt.show()

**What to notice:** the teal points sit *on* the step function at each event time — the value the online store would truly have returned. The red crosses all sit at the final 220.0, information from the future. Train on the red points and offline metrics look great while production collapses.

## 4. Delayed labels — join the outcome later

Labels (chargebacks, conversions) arrive long after the prediction. Log the features **as seen at prediction time**, keyed by id, and join the label to that snapshot when it lands — never to today's recomputed features.

In [ ]:
pred_log = scratch[['txn_id', 'event_time', 'acct_30d_avg']].copy()   # snapshot at prediction
late_labels = pd.DataFrame({'txn_id': [2, 0, 4], 'label': [0, 0, 1],
                            'label_time': [100, 70, 130]})            # arrive later, out of order
train = pred_log.merge(late_labels, on='txn_id', how='inner')
assert (train['label_time'] >= train['event_time']).all(), 'labels must post-date the prediction'
print(train)
print('trained on point-in-time features + the eventual label — no leakage ✓')

## 5. Tradeoffs & when to use it

- **One feature definition, two stores.** Compute a feature once; serve from an **online** store and train from a mirrored **offline** store to avoid skew.
- **Freshness is a deliberate axis.** Make real-time only the features that decay fast (transaction velocity); batch is fine for slow ones (account age). Fresher costs infra and PIT-correctness risk.
- **Delayed labels** force feature snapshotting; until labels arrive, unsupervised `P(X)` drift is your early warning.
- **Monitor and retrain continuously** — shadow/canary a refreshed model before promoting it.

## 6. Your turn

### Exercise 1 — Classify feature freshness tiers

Given three features, return a dict mapping each to `'real-time'`, `'near-real-time'`, or `'batch'` based on how fast it decays. A fraud-velocity feature needs real-time; a windowed average is near-real-time; a daily account-age is batch.

In [ ]:
def freshness_tier(features):
    """Map each feature name to its required freshness tier."""
    tiers = {}
    for name in features:
        # TODO(you): decide the tier from the name/semantics below
        #   'txn_velocity_1min' -> 'real-time'
        #   'acct_30d_avg'      -> 'near-real-time'
        #   'acct_age_days'     -> 'batch'
        tiers[name] = ...
    return tiers


In [ ]:
# Checks — run me
res = freshness_tier(['txn_velocity_1min', 'acct_30d_avg', 'acct_age_days'])
assert res['txn_velocity_1min'] == 'real-time', 'velocity decays in seconds'
assert res['acct_30d_avg'] == 'near-real-time', 'a 30d average is stable over minutes'
assert res['acct_age_days'] == 'batch', 'account age changes once per day'
print('✅ Exercise 1 passed')

<details>
<summary>💡 Show solution</summary>

```python
def freshness_tier(features):
    rules = {
        'txn_velocity_1min': 'real-time',
        'acct_30d_avg': 'near-real-time',
        'acct_age_days': 'batch',
    }
    return {name: rules[name] for name in features}
```

</details>

## 7. Key takeaways

- A streaming pipeline: durable **log** → **stream processor** → dual **online/offline feature store** → serving/online learning → monitoring.
- **Training/serving skew** is the top failure; **point-in-time-correct** features (as-of joins) and one shared feature definition prevent it.
- **Delayed labels**: snapshot features at prediction time, join the label later.
- **Freshness** is a deliberate cost/accuracy trade; monitor and retrain continuously.
- Back to the [course overview](https://ml-viz-ruby.vercel.app/courses/streaming-ml).